# 01 — Metrics: collection & visualization

End-to-end walk-through of Prometheus / Mimir metrics from inside the `jupyter-monitor-tools` container.

What you'll do:

1. Connect to a Prometheus-compatible HTTP API (Prometheus, Mimir, GMP, Thanos — anything that speaks the `/api/v1/query*` dialect).
2. Pull instant and range queries into `pandas.DataFrame`.
3. Plot CPU, memory, and a percentile to validate that ingestion is working.
4. Use the bundled `mimirtool` to inspect Mimir-side rules.

Set the env vars below before running. Inside the published Helm chart these come from the chart's Secret; locally export them in `.env`.

In [ ]:
import os

PROM_URL = os.environ.get('PROM_URL', os.environ.get('MIMIR_ADDRESS', 'http://prometheus:9090'))
PROM_TENANT = os.environ.get('MIMIR_TENANT_ID', '')   # blank for vanilla Prometheus
PROM_TOKEN = os.environ.get('PROM_TOKEN', '')         # bearer token if any
STEP = '30s'
RANGE_HOURS = 1

In [ ]:
import time, requests, pandas as pd

def _headers():
    h = {}
    if PROM_TENANT:
        h['X-Scope-OrgID'] = PROM_TENANT
    if PROM_TOKEN:
        h['Authorization'] = f'Bearer {PROM_TOKEN}'
    return h

def query_instant(q):
    r = requests.get(f'{PROM_URL}/api/v1/query', params={'query': q}, headers=_headers(), timeout=30)
    r.raise_for_status()
    return r.json()['data']['result']

def query_range(q, hours=RANGE_HOURS, step=STEP):
    end = int(time.time())
    start = end - hours * 3600
    r = requests.get(f'{PROM_URL}/api/v1/query_range',
                     params={'query': q, 'start': start, 'end': end, 'step': step},
                     headers=_headers(), timeout=60)
    r.raise_for_status()
    out = []
    for series in r.json()['data']['result']:
        for ts, val in series['values']:
            out.append({**series['metric'], 't': pd.to_datetime(ts, unit='s'), 'v': float(val)})
    return pd.DataFrame(out)

# Smoke-test connectivity
query_instant('up')[:3]

## CPU usage by job (last hour)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df = query_range('sum by (job) (rate(process_cpu_seconds_total[5m]))')
if df.empty:
    print('No data — check PROM_URL / metric name')
else:
    pivot = df.pivot_table(index='t', columns='job', values='v', aggfunc='mean')
    pivot.plot(figsize=(11, 4), title='CPU seconds/s by job')
    plt.tight_layout()

## Memory — a top-10 snapshot

In [ ]:
rows = query_instant('topk(10, process_resident_memory_bytes)')
if rows:
    snap = pd.DataFrame([{**r['metric'], 'bytes': float(r['value'][1])} for r in rows])
    snap['MiB'] = (snap['bytes'] / 1024 / 1024).round(1)
    snap.sort_values('MiB', ascending=False)[['job', 'instance', 'MiB']].head(10)

## Histograms — p95 latency from a histogram metric

Adjust the metric name to whatever your stack exports.

In [ ]:
p95 = query_range('histogram_quantile(0.95, sum by (le, job) (rate(prometheus_http_request_duration_seconds_bucket[5m])))')
if not p95.empty:
    p95.pivot_table(index='t', columns='job', values='v').plot(figsize=(11, 4), title='p95 HTTP latency by job (s)')
    plt.tight_layout()

## Inspect rules in Mimir with `mimirtool`

The image bundles `mimirtool`. List the rule groups uploaded by this monitor-tools deployment.

In [ ]:
!MIMIR_ADDRESS=$PROM_URL MIMIR_TENANT_ID=$PROM_TENANT mimirtool rules list 2>&1 | head -40

### Where to go next

- Wire the same queries into a Grafana dashboard via `grizzly` (see `apply-grizzly-grafana-dashboards`).
- Promote a query to a recording rule under a mixin's `prometheusRules` block.
- Open `04-community-mixins.ipynb` to browse what's already vendored in this workspace.